In [ ]:
pip install torch transformers datasets peft trl bitsandbytes pandas accelerate "torchao==0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.18.0
    Uninstalling torchao-0.18.0:
      Successfully uninstalled torchao-0.18.0


In [ ]:
# Define OUTPUT_DIR and SYSTEM_PROMPT to ensure they are available for inference
OUTPUT_DIR = "./diabetes_gemma_lora"
SYSTEM_PROMPT = """You are a domain-specific assistant for Type 2 Diabetes self-management support.
GUIDELINES:
1. REFUSE DIAGNOSIS: Do not provide formal medical diagnoses or prescribe new medications.
2. ESCALATE EMERGENCIES: For acute symptoms (e.g., severe hypoglycemia, DKA), urge the user to seek immediate emergency medical care.
3. SCOPE HUMILITY: Ground answers in recognized clinical guidelines (ADA, WHO/IDF) with traceable context.
4. COMMUNAL ACCOUNTABILITY: Provide clear, shareable advice meant to be discussed with caregivers or community health workers."""

In [ ]:
from huggingface_hub import login

# Option A: Interactive prompt (safest - paste your token when prompted)
login()

# Option B: Pass your token string directly
# login(token="hf_YOUR_NEW_TOKEN_HERE")
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, TaskType
from trl import SFTTrainer

# Set environment variable to help with CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ==========================================
# 1. SYSTEM PROMPT & GEMMA MODEL ID
# ==========================================
# Note: Gemma 2 uses <start_of_turn>user ... <end_of_turn> and <start_of_turn>model ... <end_of_turn> formatting.
MODEL_ID = "google/gemma-2b-it"  # Changed to the original Gemma 2B model for lower memory footprint
OUTPUT_DIR = "./diabetes_gemma_lora"

SYSTEM_PROMPT = """You are a domain-specific assistant for Type 2 Diabetes self-management support.
GUIDELINES:
1. REFUSE DIAGNOSIS: Do not provide formal medical diagnoses or prescribe new medications.
2. ESCALATE EMERGENCIES: For acute symptoms (e.g., severe hypoglycemia, DKA), urge the user to seek immediate emergency medical care.
3. SCOPE HUMILITY: Ground answers in recognized clinical guidelines (ADA, WHO/IDF) with traceable context.
4. COMMUNAL ACCOUNTABILITY: Provide clear, shareable advice meant to be discussed with caregivers or community health workers."""

# ==========================================
# 2. DATA PREPARATION FOR GEMMA FORMATTING
# ==========================================
def load_and_format_data():
    formatted_records = []

    # A. Process diabetes_QA_dataset.csv
    if os.path.exists('diabetes_QA_dataset.csv'):
        qa_df = pd.read_csv('diabetes_QA_dataset.csv')
        for _, row in qa_df.iterrows():
            formatted_records.append({
                "instruction": str(row['question']),
                "input": "",
                "output": str(row['answer'])
            })

    # B. Process diabetes_instruct_temp_v44.csv
    if os.path.exists('diabetes_instruct_temp_v44.csv'):
        v44_df = pd.read_csv('diabetes_instruct_temp_v44.csv')
        for _, row in v44_df.iterrows():
            formatted_records.append({
                "instruction": str(row['instruction']),
                "input": str(row['input']) if pd.notna(row['input']) else "",
                "output": str(row['output'])
            })

    # C. Process ada_diabetes_5000_instruction.csv
    if os.path.exists('ada_diabetes_5000_instruction.csv'):
        ada_df = pd.read_csv('ada_diabetes_5000_instruction.csv')
        for _, row in ada_df.iterrows():
            formatted_records.append({
                "instruction": str(row['instruction']),
                "input": str(row['input']) if pd.notna(row['input']) else "",
                "output": str(row['output'])
            })

    # Apply Gemma Prompt Structure: <start_of_turn>user ... <end_of_turn> <start_of_turn>model ... <end_of_turn>
    formatted_prompts = []
    for item in formatted_records:
        user_message = item['instruction']
        if item['input']:
            user_message += f"\n\nContext:\n{item['input']}"

        full_text = (
            f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\nUser Query: {user_message}<end_of_turn>\n"
            f"<start_of_turn>model\n{item['output']}<end_of_turn>"
        )
        formatted_prompts.append({"text": full_text})

    print(f"Total training samples prepared for Gemma: {len(formatted_prompts)}")
    return Dataset.from_pandas(pd.DataFrame(formatted_prompts))

# ==========================================
# 3. MODEL, TOKENIZER & QLORA CONFIG
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Clear CUDA cache before loading model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 'cpu'} # Changed to explicitly offload to CPU
)

# Targeting Gemma Attention & MLP Modules
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# ==========================================
# 4. EXECUTE TRAINING
# ==========================================
if __name__ == "__main__":
    dataset = load_and_format_data()

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1, # Reduced batch size
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=0.1,
        save_strategy="epoch",
        bf16=False,
        optim="paged_adamw_8bit",
        report_to="none",
        gradient_checkpointing=True # Enabled gradient checkpointing
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        args=training_args
    )

    print("Starting Gemma LoRA fine-tuning...")
    trainer.train()

    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Gemma adapters successfully saved to {OUTPUT_DIR}")


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Truncating train dataset:   0%|          | 0/6192 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/6192 [00:00<?, ? examples/s]

Starting Gemma LoRA fine-tuning...


Step,Training Loss
10,1.516167
20,0.215249
30,0.149305
40,0.181473
50,0.149449
60,0.100576
70,0.121223


Gemma adapters successfully saved to ./diabetes_gemma_lora


### Run Inference with the Fine-tuned Gemma Model

First, we'll load the tokenizer and the fine-tuned model from the `OUTPUT_DIR`. We'll use the `AutoTokenizer` and `AutoModelForCausalLM` classes from the `transformers` library.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the tokenizer and model from the fine-tuned output directory
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, device_map="auto") # Ensure model is loaded onto available device

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Next, we'll define a function to construct the prompt in the format expected by the Gemma model, including the `SYSTEM_PROMPT` for context. Then, we'll use this function to generate a response.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SYSTEM_PROMPT = """You are a domain-specific assistant for Type 2 Diabetes self-management support.
GUIDELINES:
1. REFUSE DIAGNOSIS: Do not provide formal medical diagnoses or prescribe new medications.
2. ESCALATE EMERGENCIES: For acute symptoms (e.g., severe hypoglycemia, DKA), urge the user to seek immediate emergency medical care.
3. SCOPE HUMILITY: Ground answers in recognized clinical guidelines (ADA, WHO/IDF) with traceable context.
4. COMMUNAL ACCOUNTABILITY: Provide clear, shareable advice meant to be discussed with caregivers or community health workers."""

def generate_response(question):
    # Format the prompt using the Gemma structure and system prompt
    user_message = f"{SYSTEM_PROMPT}\n\nUser Query: {question}"
    chat_template = f"<start_of_turn>user\n{user_message}<end_of_turn>\n<start_of_turn>model\n"

    # Encode the input and generate a response
    input_ids = tokenizer.encode(chat_template, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=256, # Limit the length of the generated response
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode the generated text and extract the model's response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the model's response part (after the last <start_of_turn>model\n)
    response_start_tag = "<start_of_turn>model\n"
    if response_start_tag in generated_text:
        model_response = generated_text.split(response_start_tag)[-1].strip()
    else:
        model_response = generated_text # Fallback if tag not found

    return model_response

# Example Usage:
question = "What are some healthy snack ideas for someone with Type 2 Diabetes?"
print(f"User: {question}")
response = generate_response(question)
print(f"Model: {response}")

In [ ]:
import shutil

# Define the destination path in Google Drive
# You might want to change 'MyDrive/fine_tuned_gemma_model' to your preferred path
drive_path = '/content/drive/MyDrive/fine_tuned_gemma_model'

# Copy the entire output directory to Google Drive
shutil.copytree(OUTPUT_DIR, drive_path, dirs_exist_ok=True)

print(f"Fine-tuned model saved to: {drive_path}")

Fine-tuned model saved to: /content/drive/MyDrive/fine_tuned_gemma_model


In [ ]:
import pandas as pd

# Load the evaluation dataset
qa_eval_df = pd.read_csv('diabetes_QA_dataset.csv')

# Initialize lists to store questions, ground truth answers, and model responses
eval_questions = []
eval_ground_truth_answers = []
eval_model_responses = []

# Iterate through the dataset and generate responses
for index, row in qa_eval_df.iterrows():
    question = str(row['question'])
    ground_truth_answer = str(row['answer'])

    model_response = generate_response(question)

    eval_questions.append(question)
    eval_ground_truth_answers.append(ground_truth_answer)
    eval_model_responses.append(model_response)

# Create a DataFrame for evaluation results
eval_results_df = pd.DataFrame({
    'question': eval_questions,
    'ground_truth_answer': eval_ground_truth_answers,
    'model_response': eval_model_responses
})

print("Evaluation responses generated.")

### Sample Evaluation Results

Below is a sample of the questions, ground truth answers from the dataset, and the responses generated by the fine-tuned Gemma model. This allows for a qualitative assessment of the model's performance.

In [ ]:
# Display a sample of the evaluation results
num_samples_to_display = 5

display(eval_results_df.head(num_samples_to_display))

In [ ]:
print('hello world')

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,        # Reduce batch size to 1
    gradient_accumulation_steps=8,        # Keep effective batch size at 8 (1 * 8)
    gradient_checkpointing=True,          # Saves huge amounts of activation memory
    fp16=True,                            # Use mixed precision (or bf16=True if using A100)
    # ... your other args
)

In [ ]:
pip install trl[peft]